In [2]:
# Importing ChexFound Code Base

import os
import sys
import importlib

# 1. Setup repository path
repo_dir = '/kaggle/working/CheXFound'
if not os.path.exists(os.path.join(repo_dir, 'chexfound', '__init__.py')):
    print("Cloning CheXFound codebase...")
    !rm -rf /kaggle/working/CheXFound
    !git clone https://github.com/SuyogMagar/CheXFound.git /kaggle/working/CheXFound

# 2. Register path permanently for this notebook session
if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)

importlib.invalidate_caches()

# 3. Test import to guarantee future cells will not fail
from chexfound.models import vision_transformer as vits
print("\n✓ SUCCESS: CheXFound codebase is imported, verified, and locked in sys.path.")


✓ SUCCESS: CheXFound codebase is imported, verified, and locked in sys.path.


In [3]:
# Dataset Verification & Inspection: JSRT (Lung Nodules)

import os
import glob
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# 1. Directly index JSRT image files across input folders
# Standard JSRT nomenclature:
# 'JPCLN' -> Japanese Pulmonary Cancer / Lung Nodule (Positive: 1)
# 'JPCNN' -> Japanese Pulmonary Cancer Non-Nodule / Normal (Negative: 0)
all_files = glob.glob('/kaggle/input/**/JPCL*.*', recursive=True) + glob.glob('/kaggle/input/**/JPCN*.*', recursive=True)
all_files = [p for p in all_files if p.lower().endswith(('.png', '.jpg', '.jpeg'))]

assert len(all_files) > 0, "Could not find JSRT images (JPCLN/JPCNN) in /kaggle/input/"

records = []
seen = set()
for p in all_files:
    fname = os.path.basename(p)
    if fname in seen:
        continue
    seen.add(fname)
    if 'JPCLN' in fname:
        label = 1
    elif 'JPCNN' in fname:
        label = 0
    else:
        continue
    records.append({
        'study_id': fname,
        'Nodule': label,
        'full_path': p
    })

df = pd.DataFrame(records).sort_values('study_id').reset_index(drop=True)

# 2. Optional: Merge with JSRT metadata CSV if available
meta_csvs = glob.glob('/kaggle/input/**/jsrt_metadata.csv', recursive=True)
if len(meta_csvs) > 0:
    meta_df = pd.read_csv(meta_csvs[0])
    print(f"Found Metadata CSV: {meta_csvs[0]}")
    # Normalize study_id extensions for clean inner/left join
    meta_df['norm_id'] = meta_df['study_id'].apply(lambda x: str(x) if str(x).endswith(('.png', '.jpg')) else f"{x}.png")
    df['norm_id'] = df['study_id'].apply(lambda x: str(x) if str(x).endswith(('.png', '.jpg')) else f"{x}.png")
    df = df.merge(meta_df.drop(columns=['study_id'], errors='ignore'), on='norm_id', how='left').drop(columns=['norm_id'])

# 3. Paper Section IV-B split (70:10:20 -> 172 Train, 25 Val, 50 Test)
indices = np.arange(len(df))
train_val_idx, test_idx = train_test_split(
    indices, test_size=50, random_state=42, stratify=df['Nodule']
)
train_idx, val_idx = train_test_split(
    train_val_idx, test_size=25, random_state=42, stratify=df.loc[train_val_idx, 'Nodule']
)

df['split'] = 'train'
df.loc[val_idx, 'split'] = 'val'
df.loc[test_idx, 'split'] = 'test'

train_df = df[df['split'] == 'train'].reset_index(drop=True)
val_df = df[df['split'] == 'val'].reset_index(drop=True)
test_df = df[df['split'] == 'test'].reset_index(drop=True)

print(f"\n✓ Verified JSRT Scans: {len(df)}")
print(f"✓ Normal: {(df['Nodule'] == 0).sum()} | Nodule: {(df['Nodule'] == 1).sum()}")
print(f"✓ Train Pool: {len(train_df)} | Val Pool: {len(val_df)} | Test Pool: {len(test_df)}")

# Display head of processed JSRT dataset
cols_to_show = ['study_id', 'Nodule', 'split', 'full_path']
if 'subtlety' in df.columns:
    cols_to_show.insert(2, 'subtlety')
df[cols_to_show].head(5)

Found Metadata CSV: /kaggle/input/datasets/raddar/nodules-in-chest-xrays-jsrt/jsrt_metadata.csv

✓ Verified JSRT Scans: 247
✓ Normal: 93 | Nodule: 154
✓ Train Pool: 172 | Val Pool: 25 | Test Pool: 50


,study_id,Nodule,subtlety,split,full_path
0,JPCLN001.png,1,5.0,test,/kaggle/input/datasets/raddar/nodules-in-chest...
1,JPCLN002.png,1,5.0,train,/kaggle/input/datasets/raddar/nodules-in-chest...
2,JPCLN003.png,1,5.0,val,/kaggle/input/datasets/raddar/nodules-in-chest...
3,JPCLN004.png,1,5.0,train,/kaggle/input/datasets/raddar/nodules-in-chest...
4,JPCLN005.png,1,5.0,test,/kaggle/input/datasets/raddar/nodules-in-chest...


In [4]:
# Load and Freeze the ChexFound Trained Parameters

import os
import glob
import torch
import torch.nn as nn
from chexfound.models import vision_transformer as vits

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device for inference: {device}")

# 1. Locate teacher checkpoint
teacher_files = glob.glob('/kaggle/input/**/teacher_checkpoint.pth', recursive=True)
assert len(teacher_files) > 0, "Could not find teacher_checkpoint.pth in /kaggle/input/"
teacher_path = teacher_files[0]
print(f"Loading weights from: {teacher_path}")

# 2. Instantiate ViT-Large backbone (512x512 resolution, patch_size=16)
model = vits.vit_large(patch_size=16, img_size=512)

# 3. Load checkpoint and align state dict keys
teacher_ckpt = torch.load(teacher_path, map_location='cpu')
state_dict = teacher_ckpt.get('teacher', teacher_ckpt)
clean_state_dict = {
    (k[9:] if k.startswith('backbone.') else k): v 
    for k, v in state_dict.items()
}

missing_keys, unexpected_keys = model.load_state_dict(clean_state_dict, strict=False)

# 4. Freeze backbone and mount on GPU
model.eval()
model.to(device)
for param in model.parameters():
    param.requires_grad = False

print(f"\n✓ CheXFound ViT-Large Loaded:")
print(f"  - Missing keys: {len(missing_keys)}")
print(f"  - Unexpected keys: {len(unexpected_keys)}")
print(f"  - All parameters frozen: {all(not p.requires_grad for p in model.parameters())}")

# 5. Sanity check: Run a dummy forward pass through the last 4 blocks
dummy = torch.randn(1, 3, 512, 512).to(device)
with torch.no_grad():
    sample_layers = model.get_intermediate_layers(dummy, n=4, return_class_token=True)
    # Each block outputs: (patch_tokens [B, 1024, 1024], cls_token [B, 1024])
    patch_sample, cls_sample = sample_layers[-1]
    
print(f"\n✓ Forward check successful:")
print(f"  - Patch tokens shape: {patch_sample.shape} (1024 spatial patches)")
print(f"  - CLS token shape   : {cls_sample.shape} (1024 dimensions)")

Device for inference: cuda
Loading weights from: /kaggle/input/datasets/suyogmagar08/chexfound-weights/teacher_checkpoint.pth

✓ CheXFound ViT-Large Loaded:
  - Missing keys: 240
  - Unexpected keys: 305
  - All parameters frozen: True

✓ Forward check successful:
  - Patch tokens shape: torch.Size([1, 1024, 1024]) (1024 spatial patches)
  - CLS token shape   : torch.Size([1, 1024]) (1024 dimensions)


In [1]:
import os
import sys
import glob
import re
import importlib
import subprocess
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
from torchvision.transforms import InterpolationMode
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# 1. Setup CheXFound Repository
repo_dir = '/kaggle/working/CheXFound'
if not os.path.exists(os.path.join(repo_dir, 'chexfound', '__init__.py')):
    !rm -rf /kaggle/working/CheXFound
    !git clone https://github.com/RPIDIAL/CheXFound.git /kaggle/working/CheXFound || git clone https://github.com/SuyogMagar/CheXFound.git /kaggle/working/CheXFound

if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)

importlib.invalidate_caches()
from chexfound.models import vision_transformer as vits

# 2. Index JSRT directly from filesystem (154 JPCLN Nodule, 93 JPCNN Normal)
cache_file = '/kaggle/working/jsrt_unified_cache.pt'

if os.path.exists(cache_file):
    print(f"Loading existing cache from {cache_file}...")
    bundle = torch.load(cache_file, map_location='cpu')
    full_layers = bundle['full_layers']
    y_all = bundle['y_all']
    y_np = y_all.numpy()
    print(f"✓ Loaded {len(full_layers)} blocks | Scans: {len(y_all)} (Nodule: {(y_np==1).sum()}, Normal: {(y_np==0).sum()})")
else:
    print("Indexing JSRT files from /kaggle/input/...")
    # Find all JSRT images (JPCLN = Nodule, JPCNN = Non-nodule)
    all_files = glob.glob('/kaggle/input/**/JPCL*.*', recursive=True) + glob.glob('/kaggle/input/**/JPCN*.*', recursive=True)
    all_files = [p for p in all_files if p.lower().endswith(('.png', '.jpg', '.jpeg'))]

    records = []
    seen = set()
    for p in all_files:
        fname = os.path.basename(p)
        if fname in seen:
            continue
        seen.add(fname)
        if 'JPCLN' in fname:
            label = 1
        elif 'JPCNN' in fname:
            label = 0
        else:
            continue
        records.append({'filename': fname, 'path': p, 'label': label})

    df_jsrt = pd.DataFrame(records).sort_values('filename').reset_index(drop=True)
    y_all = torch.tensor(df_jsrt['label'].values)
    y_np = y_all.numpy()
    print(f"✓ Indexed {len(df_jsrt)} total scans (Nodule [JPCLN]: {(y_np==1).sum()}, Normal [JPCNN]: {(y_np==0).sum()})")
    assert len(df_jsrt) == 247, f"Expected 247 images, found {len(df_jsrt)}"

    # 3. Load Backbone and Extract Features
    teacher_files = glob.glob('/kaggle/input/**/teacher_checkpoint.pth', recursive=True)
    assert len(teacher_files) > 0, "teacher_checkpoint.pth missing!"
    teacher_ckpt = torch.load(teacher_files[0], map_location='cpu')
    raw_state = teacher_ckpt.get('teacher', teacher_ckpt)

    clean_state = {}
    for k, v in raw_state.items():
        if k.startswith('dino_head.') or k.startswith('ibot_head.'):
            continue
        new_k = k
        for pfx in ['module.', 'backbone.']:
            if new_k.startswith(pfx):
                new_k = new_k[len(pfx):]
        new_k = re.sub(r'^blocks\.\d+\.(\d+)\.', r'blocks.\1.', new_k)
        clean_state[new_k] = v

    has_swiglu = any('w12' in k for k in clean_state.keys())
    has_layerscale = any('ls1' in k for k in clean_state.keys())
    has_registers = 'register_tokens' in clean_state
    num_registers = clean_state['register_tokens'].shape[1] if has_registers else 0
    pos_shape = clean_state['pos_embed'].shape
    native_res = int(((pos_shape[1] - 1) ** 0.5) * 16)

    model = vits.vit_large(
        patch_size=16,
        img_size=native_res,
        init_values=1e-5 if has_layerscale else None,
        ffn_layer="swiglu" if has_swiglu else "mlp",
        block_chunks=0,
        num_register_tokens=num_registers
    )
    msg = model.load_state_dict(clean_state, strict=False)
    assert len(msg.missing_keys) == 0, f"Backbone missing keys: {msg.missing_keys}"
    print("✓ CheXFound ViT-Large backbone loaded (0 missing keys)")

    model.eval().to(device)
    for p in model.parameters():
        p.requires_grad = False

    eval_transform = T.Compose([
        T.Resize((512, 512), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    class JSRTDataset(Dataset):
        def __init__(self, df, transform):
            self.df = df
            self.transform = transform
        def __len__(self):
            return len(self.df)
        def __getitem__(self, idx):
            img = Image.open(self.df.iloc[idx]['path']).convert('RGB')
            return self.transform(img), self.df.iloc[idx]['label']

    loader = DataLoader(JSRTDataset(df_jsrt, eval_transform), batch_size=16, shuffle=False, num_workers=2)

    batch_layers = []
    with torch.no_grad():
        for imgs, _ in tqdm(loader, desc="Extracting JSRT Layers (~40s)"):
            imgs = imgs.to(device)
            layers = model.get_intermediate_layers(imgs, n=4, return_class_token=True)
            batch_layers.append([(p.cpu(), c.cpu()) for p, c in layers])

    full_layers = []
    for b in range(4):
        p_b = torch.cat([batch[b][0] for batch in batch_layers], dim=0)
        c_b = torch.cat([batch[b][1] for batch in batch_layers], dim=0)
        full_layers.append((p_b, c_b))

    torch.save({'full_layers': full_layers, 'y_all': y_all}, cache_file)
    print(f"✓ Saved verified tokens to {cache_file}")

# 4. Paper Split: 70% Train (172), 10% Val (25), 20% Test (50)
y_np = y_all.numpy()
indices = np.arange(len(y_np))
train_val_idx, test_idx = train_test_split(indices, test_size=50, random_state=42, stratify=y_np)
train_idx, val_idx = train_test_split(train_val_idx, test_size=25, random_state=42, stratify=y_np[train_val_idx])

print(f"\nSplit Sizes -> Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}")

# =========================================================================
# BENCHMARK 1: LINEAR PROBE (8192-dim: 4x CLS + 4x AvgPool Patches)
# =========================================================================
X_cls = torch.cat([full_layers[b][1] for b in range(4)], dim=-1)
X_avg = torch.cat([full_layers[b][0].mean(dim=1) for b in range(4)], dim=-1)
X_comb = torch.cat([X_cls, X_avg], dim=-1)
X_comb_norm = F.normalize(X_comb, p=2, dim=-1).numpy()

X_tr, y_tr = X_comb_norm[train_idx], y_np[train_idx]
X_va, y_va = X_comb_norm[val_idx], y_np[val_idx]
X_te, y_te = X_comb_norm[test_idx], y_np[test_idx]

best_c, best_val_lp = 1.0, -1.0
for c_val in [0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 20.0]:
    clf = LogisticRegression(C=c_val, max_iter=1000, solver='lbfgs')
    clf.fit(X_tr, y_tr)
    v_auc = roc_auc_score(y_va, clf.predict_proba(X_va)[:, 1])
    if v_auc > best_val_lp:
        best_val_lp = v_auc
        best_c = c_val

X_tr_val = np.vstack([X_tr, X_va])
y_tr_val = np.concatenate([y_tr, y_va])
lp_clf = LogisticRegression(C=best_c, max_iter=1000, solver='lbfgs')
lp_clf.fit(X_tr_val, y_tr_val)
lp_test_probs = lp_clf.predict_proba(X_te)[:, 1]

lp_test_auprc = average_precision_score(y_te, lp_test_probs)
lp_test_auroc = roc_auc_score(y_te, lp_test_probs)

print("\n" + "="*70)
print("     TABLE IV BENCHMARK 1: LINEAR PROBE (JSRT)")
print("="*70)
print(f"Val AUROC  (25 scans)  : {best_val_lp:.4f} (Best C={best_c})")
print(f"Test AUPRC (50 scans)  : {lp_test_auprc:.4f}  (Table IV Target: 0.918 [0.826 - 0.975])")
print(f"Test AUROC (50 scans)  : {lp_test_auroc:.4f}  (Table IV Target: 0.856 [0.741 - 0.948])")
print("="*70)

# =========================================================================
# BENCHMARK 2: GLoRI CROSS-ATTENTION HEAD (Paper Protocol: Batch Size 16)
# =========================================================================
class PyramidPatchMerging(nn.Module):
    def __init__(self, embed_dim=1024, out_dim=768):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(embed_dim, out_dim),
            nn.ReLU(),
            nn.LayerNorm(out_dim)
        )
    def forward(self, patch_tokens):
        B, N, D = patch_tokens.shape
        H = W = int(N ** 0.5)
        x = patch_tokens.transpose(1, 2).view(B, D, H, W)
        p8 = F.adaptive_avg_pool2d(x, (4, 4)).flatten(2).transpose(1, 2)
        p4 = F.adaptive_avg_pool2d(x, (8, 8)).flatten(2).transpose(1, 2)
        p2 = F.adaptive_avg_pool2d(x, (16, 16)).flatten(2).transpose(1, 2)
        return self.proj(torch.cat([p8, p4, p2], dim=1))

class GLoRIHead(nn.Module):
    def __init__(self, in_dim=1024, glori_dim=768):
        super().__init__()
        self.disease_query = nn.Parameter(torch.randn(1, 1, glori_dim) * 0.02)
        self.patch_proj = nn.Sequential(nn.Linear(in_dim, glori_dim), nn.ReLU())
        self.temp_mlp = nn.Sequential(
            nn.Linear(glori_dim, glori_dim // 2),
            nn.Tanh(),
            nn.Linear(glori_dim // 2, 1)
        )
        self.mha_fine = nn.MultiheadAttention(embed_dim=glori_dim, num_heads=8, batch_first=True)
        self.pyramid = PyramidPatchMerging(embed_dim=in_dim, out_dim=glori_dim)
        self.mha_coarse = nn.MultiheadAttention(embed_dim=glori_dim, num_heads=8, batch_first=True)
        self.cls_proj = nn.Sequential(nn.Linear(in_dim, glori_dim), nn.LayerNorm(glori_dim))
        self.norm = nn.LayerNorm(glori_dim * 3)
        self.classifier = nn.Linear(glori_dim * 3, 1)

    def forward(self, layers_data):
        patch_tokens, cls_token = layers_data[-1]
        B = patch_tokens.shape[0]
        queries = self.disease_query.expand(B, -1, -1)
        proj_patches = self.patch_proj(patch_tokens)
        
        pooled = proj_patches.mean(dim=1)
        tau = torch.exp(self.temp_mlp(pooled)).unsqueeze(-1)
        queries_scaled = queries / (tau + 1e-6)
        fine_feat, _ = self.mha_fine(queries_scaled, proj_patches, proj_patches)
        
        coarse_patches = self.pyramid(patch_tokens)
        coarse_feat, _ = self.mha_coarse(queries, coarse_patches, coarse_patches)
        
        global_feat = self.cls_proj(cls_token).unsqueeze(1)
        integrated = torch.cat([fine_feat, coarse_feat, global_feat], dim=-1).squeeze(1)
        return self.classifier(self.norm(integrated))

def train_eval_glori(tr_idx, ev_idx, lr, epochs=100, batch_size=16):
    torch.manual_seed(42)
    m = GLoRIHead().to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-6)
    crit = nn.BCEWithLogitsLoss()
    
    n_tr = len(tr_idx)
    for epoch in range(epochs):
        m.train()
        perm = torch.randperm(n_tr)
        for i in range(0, n_tr, batch_size):
            b_idx = tr_idx[perm[i:i + batch_size]]
            b_layers = [(p[b_idx].to(device), c[b_idx].to(device)) for p, c in full_layers]
            b_y = y_all[b_idx].float().unsqueeze(1).to(device)
            opt.zero_grad()
            crit(m(b_layers), b_y).backward()
            opt.step()
        sched.step()
        
    m.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(ev_idx), batch_size):
            b_idx = ev_idx[i:i + batch_size]
            b_layers = [(p[b_idx].to(device), c[b_idx].to(device)) for p, c in full_layers]
            preds.append(np.atleast_1d(torch.sigmoid(m(b_layers)).cpu().squeeze(-1).numpy()))
    return np.concatenate(preds)

print("\n--- Stage 1: GLoRI Validation Learning Rate Search (Batch Size 16) ---")
best_glori_lr, best_val_glori = 1e-4, -1.0
for lr in [1e-5, 2e-5, 5e-5, 1e-4, 2e-4]:
    v_preds = train_eval_glori(train_idx, val_idx, lr=lr, epochs=70, batch_size=16)
    v_auc = roc_auc_score(y_np[val_idx], v_preds)
    print(f"GLoRI LR {lr:8.1e} | Val AUROC: {v_auc:.4f}")
    if v_auc > best_val_glori:
        best_val_glori = v_auc
        best_glori_lr = lr

print(f"\n✓ Selected Best GLoRI LR: {best_glori_lr} (Val AUROC: {best_val_glori:.4f})")
print("--- Stage 2: Retraining GLoRI on Train+Val (197 scans) & Testing (50 scans) ---")
glori_test_probs = train_eval_glori(train_val_idx, test_idx, lr=best_glori_lr, epochs=100, batch_size=16)
glori_test_auprc = average_precision_score(y_np[test_idx], glori_test_probs)
glori_test_auroc = roc_auc_score(y_np[test_idx], glori_test_probs)

print("\n" + "="*70)
print("     TABLE IV BENCHMARK 2: GLoRI HEAD (JSRT)")
print("="*70)
print(f"GLoRI Test AUPRC (50 scans) : {glori_test_auprc:.4f}  (Table IV Target: 0.986 [0.956 - 1.0])")
print(f"GLoRI Test AUROC (50 scans) : {glori_test_auroc:.4f}  (Table IV Target: 0.975 [0.931 - 1.0])")
print("="*70)

Device: cuda
Cloning into '/kaggle/working/CheXFound'...
remote: Enumerating objects: 460, done.
remote: Counting objects: 100% (460/460), done.
remote: Compressing objects: 100% (358/358), done.
remote: Total 460 (delta 134), reused 407 (delta 96), pack-reused 0 (from 0)
Receiving objects: 100% (460/460), 24.14 MiB | 33.40 MiB/s, done.
Resolving deltas: 100% (134/134), done.
Indexing JSRT files from /kaggle/input/...


/kaggle/working/CheXFound/chexfound/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/kaggle/working/CheXFound/chexfound/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/kaggle/working/CheXFound/chexfound/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


✓ Indexed 247 total scans (Nodule [JPCLN]: 154, Normal [JPCNN]: 93)
✓ CheXFound ViT-Large backbone loaded (0 missing keys)


Extracting JSRT Layers (~40s): 100%|██████████| 16/16 [01:07<00:00,  4.25s/it]


✓ Saved verified tokens to /kaggle/working/jsrt_unified_cache.pt

Split Sizes -> Train: 172 | Val: 25 | Test: 50

     TABLE IV BENCHMARK 1: LINEAR PROBE (JSRT)
Val AUROC  (25 scans)  : 0.8681 (Best C=20.0)
Test AUPRC (50 scans)  : 0.8528  (Table IV Target: 0.918 [0.826 - 0.975])
Test AUROC (50 scans)  : 0.7657  (Table IV Target: 0.856 [0.741 - 0.948])

--- Stage 1: GLoRI Validation Learning Rate Search (Batch Size 16) ---
GLoRI LR  1.0e-05 | Val AUROC: 0.8333
GLoRI LR  2.0e-05 | Val AUROC: 0.8333
GLoRI LR  5.0e-05 | Val AUROC: 0.8542
GLoRI LR  1.0e-04 | Val AUROC: 0.9444
GLoRI LR  2.0e-04 | Val AUROC: 0.8611

✓ Selected Best GLoRI LR: 0.0001 (Val AUROC: 0.9444)
--- Stage 2: Retraining GLoRI on Train+Val (197 scans) & Testing (50 scans) ---

     TABLE IV BENCHMARK 2: GLoRI HEAD (JSRT)
GLoRI Test AUPRC (50 scans) : 0.9185  (Table IV Target: 0.986 [0.956 - 1.0])
GLoRI Test AUROC (50 scans) : 0.8472  (Table IV Target: 0.975 [0.931 - 1.0])
